# nanoGPT GPU 性能评估

本笔记整理了在 RunPod 上使用不同 GPU 配置运行 nanoGPT 的性能分析，以及从学习视角对各配置的价值评估。

## 1. nanoGPT 项目概览

| 方面 | 说明 |
|------|------|
| 核心文件 | `train.py` (~300行训练循环), `model.py` (~300行GPT定义) |
| 默认训练任务 | GPT-2 124M on OpenWebText |
| 官方基准 | 8x A100 40GB, ~4天完成训练 |
| 多GPU支持 | PyTorch DDP (`torchrun`) |
| 默认精度 | BF16 (若硬件支持) |

官方训练命令：
```bash
# 单卡
python train.py config/train_gpt2.py

# 多卡 (如 4 张)
torchrun --standalone --nproc_per_node=4 train.py config/train_gpt2.py
```

> **注意**: nanoGPT README (Nov 2025) 提示该项目已 deprecated，新版为 [nanochat](https://github.com/karpathy/nanochat)。但用于学习 Transformer 原理，nanoGPT 仍是极佳材料。

## 2. GPU 规格对比

| GPU | BF16算力 (Dense) | 显存 | 内存带宽 | 互联 | TDP |
|-----|----------------|------|---------|------|-----|
| RTX 5090 | ~419 TFLOPS | 32 GB GDDR7 | 1.79 TB/s | PCIe 5.0 | 575W |
| RTX 4090 | ~330 TFLOPS | 24 GB GDDR6X | 1.008 TB/s | PCIe 4.0 | 450W |
| A100 40GB (参考) | ~312 TFLOPS | 40 GB HBM2e | 2.0 TB/s | NVLink 3.0 | 400W |

**关键差异**：消费级 GPU (4090/5090) 无 NVLink，多卡通过 PCIe 通信，延迟远高于服务器 GPU 的 NVLink 互联。

## 3. 各配置性能估算

### 3.1 GPT-2 124M 完整训练 (600K iterations, OpenWebText)

| 配置 | 理论算力 | 多卡通信效率 | 预估训练时间 |
|------|---------|------------|------------|
| 8x A100 40GB (官方) | ~2496 TFLOPS | ~90% (NVLink) | **4 天** |
| RTX 5090 x1 | ~419 TFLOPS | 100% | **~11 天** |
| RTX 4090 x1 | ~330 TFLOPS | 100% | **~14 天** |
| RTX 4090 x2 | ~660 TFLOPS | ~75-80% (PCIe) | **~8-9 天** |
| RTX 4090 x4 | ~1320 TFLOPS | ~60-70% (PCIe) | **~5-6 天** |

### 3.2 Shakespeare Char 小模型 (A100 约 3 分钟)

| 配置 | 预估时间 | 备注 |
|------|---------|------|
| RTX 5090 x1 | ~3-4 min | |
| RTX 4090 x1 | ~4-5 min | |
| RTX 4090 x2 | ~3-4 min | 小任务 DDP 提升有限 |
| RTX 4090 x4 | ~2-3 min | |

### 3.3 RTX 5090 vs RTX 4090 单卡详析

- **算力**: +27% (419 vs 330 TFLOPS)
- **内存带宽**: +78% (1.79 vs 1.0 TB/s) — 对 attention 计算影响更大
- **显存**: +33% (32GB vs 24GB) — 可使用更大 batch_size
- **实际综合加速**: ~1.4-1.6x

## 4. 显存需求与 batch size

nanoGPT 默认配置 (`batch_size=12`, `block_size=1024`, BF16)：
- GPT-2 124M 约需 **10-14 GB 显存**
- RTX 4090 (24GB): 可适当提高 batch_size
- RTX 5090 (32GB): 可跑更大模型 (gpt2-xl 1.5B) 或更大 batch

**单卡适配 GPT-2 训练** — 等效 batch size 调整：
```python
# 原始配置 (8卡): batch_size=12 * grad_accum=40 * 8 GPUs ≈ 0.5M tokens/step
# 单卡等效配置:
gradient_accumulation_steps = 320  # 40 * 8
batch_size = 12
```

## 5. 从学习视角：x2 双卡的独特价值

**结论：x2 是学习分布式训练原理的最佳配置**，原因如下：

### 5.1 亲身体验 DDP 全部复杂性

```bash
torchrun --standalone --nproc_per_node=2 train.py config/train_gpt2.py
```

直接遇到并学习的概念：
- **NCCL All-Reduce** 如何跨卡同步梯度
- **PCIe 带宽瓶颈**的实际体感 (vs NVLink)
- `rank`, `world_size`, `local_rank` 等 DDP 核心概念
- `gradient_accumulation_steps` 与实际 batch size 的关系

### 5.2 可以做对比实验

```bash
# 实验1: 单卡，grad_accum=80 (等效 batch)
python train.py --gradient_accumulation_steps=80

# 实验2: 双卡，grad_accum=40 (同等 token/step)
torchrun --nproc_per_node=2 train.py --gradient_accumulation_steps=40
```

两者 token/step 相同，速度不同 —— 直接测量通信开销的代价。

### 5.3 bench.py 揭示 MFU 差异

```bash
# 单卡 MFU
python bench.py

# 双卡时每张卡的 MFU 会下降 (通信占用时间)
torchrun --nproc_per_node=2 bench.py
```

MFU (Model FLOPs Utilization) 的下降就是 PCIe 通信开销的直接体现。

### 5.4 各配置学习体验对比

| 配置 | 主要学习内容 | 难度 |
|------|------------|------|
| x1 | 单机训练、显存优化、`torch.compile`、梯度累积 | 低 |
| **x2** | **DDP 原理、通信开销、多进程同步、scaling efficiency** | 中 |
| x4 | x2 的延伸，更多工程调试 | 高 |

### 5.5 x2 特别适合研究的问题

**通信与计算的 overlap**：nanoGPT 的 DDP 在 backward pass 时自动 overlap 梯度通信和计算。x2 时这个效果清晰可测。

**Scaling efficiency 实测**：
```
理论: 2x 算力
实测: 1.6-1.8x 速度  ← 这个差距就是分布式训练的学习内容
```

**DDP vs Tensor Parallelism 的切入点**：DDP 每张卡持有完整模型副本，通信发生在梯度层面。理解这一点后，再对比 Tensor Parallel (切分模型本身) 会豁然开朗。

## 6. RunPod 选购建议

| 目标 | 推荐配置 | 理由 |
|------|---------|------|
| 学习/快速实验 (Shakespeare) | RTX 4090 x1 | 性价比最高 |
| 深入学习分布式训练 | RTX 4090 x2 | 复杂度适中，现象清晰 |
| 认真复现 GPT-2 124M | RTX 4090 x4 | ~5-6天，最快消费级方案 |
| 单卡最高性能 | RTX 5090 x1 | 显存大，带宽高，但贵 |

**性价比排序** (学习视角): `4090 x1` > `4090 x2` > `5090 x1` > `4090 x4`

**速度排序**: `4090 x4` ≈ `5090 x1` > `4090 x2` > `4090 x1`

## 7. 快速上手命令

```bash
# 安装依赖
pip install torch numpy transformers datasets tiktoken wandb tqdm

# 准备 Shakespeare 数据
python data/shakespeare_char/prepare.py

# 训练小模型 (单卡, ~5min on 4090)
python train.py config/train_shakespeare_char.py

# 双卡训练 GPT-2
torchrun --standalone --nproc_per_node=2 train.py config/train_gpt2.py

# 性能测试 (输出 MFU)
python bench.py

# 验证 JSON (修改 .ipynb 后)
python3 -c "import json; json.load(open('gpu-eval.ipynb'))"
```